In [ ]:
import torch
from torch import nn
from d2l import torch as d2l

def batch_norm(X, gamma, beta, moving_mean, moving_var, eps, momentum):
    """批量归一化的核心计算函数"""
    # 通过 is_grad_enabled 判断当前模式是训练还是推理
    if not torch.is_grad_enabled():  # 推理模式
        # 直接使用移动平均得到的全局均值和方差
        X_hat = (X - moving_mean) / torch.sqrt(moving_var + eps)
    else:  # 训练模式
        assert len(X.shape) in (2, 4)
        
        if len(X.shape) == 2:  # 全连接层：在特征维上计算
            mean = X.mean(dim=0)
            var = ((X - mean) ** 2).mean(dim=0)
        else:  # 卷积层：在通道维上计算（保留维度用于广播）
            mean = X.mean(dim=(0, 2, 3), keepdim=True)
            var = ((X - mean) ** 2).mean(dim=(0, 2, 3), keepdim=True)
        
        # 使用当前批次的均值和方差进行标准化
        X_hat = (X - mean) / torch.sqrt(var + eps)
        
        # 使用移动平均更新全局均值和方差（动量法）
        moving_mean = momentum * moving_mean + (1.0 - momentum) * mean
        moving_var = momentum * moving_var + (1.0 - momentum) * var
    
    # 缩放和偏移（可学习参数）
    Y = gamma * X_hat + beta
    return Y, moving_mean.data, moving_var.data

In [ ]:
class BatchNorm(nn.Module):
    def __init__(self, num_features, num_dims):
        super(BatchNorm, self).__init__()
        
        # 根据维度确定参数形状
        if num_dims == 2:  # 全连接层：形状 (1, num_features)
            shape = (1, num_features)
        else:  # 卷积层：形状 (1, num_features, 1, 1)
            shape = (1, num_features, 1, 1)
        
        # 可学习的拉伸（gamma）和偏移（beta）参数
        self.gamma = nn.Parameter(torch.ones(shape))
        self.beta = nn.Parameter(torch.zeros(shape))
        
        # 移动平均的均值和方差（非模型参数，不需要梯度）
        self.moving_mean = torch.zeros(shape)
        self.moving_var = torch.ones(shape)
    
    def forward(self, X):
        # 确保移动平均统计量与输入在同一设备上
        if self.moving_mean.device != X.device:
            self.moving_mean = self.moving_mean.to(X.device)
            self.moving_var = self.moving_var.to(X.device)
        
        # 调用核心计算函数
        Y, self.moving_mean, self.moving_var = batch_norm(
            X, self.gamma, self.beta, self.moving_mean, self.moving_var,
            eps=1e-5, momentum=0.9
        )
        return Y

In [ ]:
# 构建带 BN 的 LeNet
net = nn.Sequential(
    # 第一个卷积块
    nn.Conv2d(1, 6, kernel_size=5),
    BatchNorm(6, num_dims=4),  # BN 在卷积后、激活前
    nn.Sigmoid(),
    nn.MaxPool2d(kernel_size=2, stride=2),
    
    # 第二个卷积块
    nn.Conv2d(6, 16, kernel_size=5),
    BatchNorm(16, num_dims=4),
    nn.Sigmoid(),
    nn.MaxPool2d(kernel_size=2, stride=2),
    
    # 展平
    nn.Flatten(),
    
    # 第一个全连接块
    nn.Linear(16 * 4 * 4, 120),
    BatchNorm(120, num_dims=2),
    nn.Sigmoid(),
    
    # 第二个全连接块
    nn.Linear(120, 84),
    BatchNorm(84, num_dims=2),
    nn.Sigmoid(),
    
    # 输出层
    nn.Linear(84, 10)
)